In [1]:
def make_cpd(node_var, prob_lookup):
    """Build TabularCPD from parent-config dictionary.""" # docstring
    parents = parent_map[node_var] # node parents
    combos = list(itertools.product(*[var_domains[p] for p in parents])) # parent configurations
    first_vals = [prob_lookup[cfg] for cfg in combos] # positive probabilities
    cpt_matrix = [first_vals, [1 - p for p in first_vals]] # binary complement matrix
    return TabularCPD( # create factor
        node_var, 2, cpt_matrix, # binary variable specification
        evidence=parents or None, # conditioning variables
        evidence_card=[len(var_domains[p]) for p in parents] or None, # card dimensions
        state_names={node_var: var_domains[node_var], **{p: var_domains[p] for p in parents}} # explicit state labels
    ) # return factor

In [2]:
def build_expert_model():
    net = DiscreteBayesianNetwork(graph_edges) # initialize network
    net.add_cpds( # attach factors
        TabularCPD("Weather", 3, [[p_w_prior[s]] for s in var_domains["Weather"]], # weather cpd
                   state_names={"Weather": var_domains["Weather"]}), # weather states
        TabularCPD("DayType", 2, [[p_d_prior[s]] for s in var_domains["DayType"]], # day cpd
                   state_names={"DayType": var_domains["DayType"]}), # day states
        make_cpd("Occupancy", p_occ_hi), # occupancy factor
        make_cpd("AC", p_ac_on), # ac factor
        make_cpd("Load", p_ld_hvy), # load factor
        make_cpd("Energy", p_en_hi), # energy factor
    ) # factors complete
    return net # return network

In [3]:
def draw_network(bn_model, save_path, plot_title):
    di_graph = nx.DiGraph(bn_model.edges()) # build directed graph
    layout_coords = {"Weather": (0, 2), "DayType": (3, 2), "Occupancy": (3, 1), # node layout coords
                     "AC": (1, 0.5), "Load": (4, 0.5), "Energy": (2.5, -0.5)} # position mappings
    plt.figure(figsize=(8, 5)) # canvas sizing
    nx.draw(di_graph, layout_coords, with_labels=True, node_color="#a8d0f0", node_size=3200, # draw structure
            font_size=10, font_weight="bold", arrowsize=22, edge_color="#444") # styling parameters
    plt.title(plot_title) # title text
    plt.savefig(save_path, dpi=150, bbox_inches="tight") # write file
    plt.close() # free memory

In [4]:
def run_queries(target_bn):
    engine = VariableElimination(target_bn) # inference engine
    results = {} # store outputs
    for q_code, (q_var, q_st, ev_map) in query_cases.items(): # iterate query pool
        factor_dist = engine.query([q_var], evidence=ev_map, show_progress=False) # marginalize factors
        results[q_code] = float(factor_dist.get_value(**{q_var: q_st})) # extract value
    return results # return dictionary

In [5]:
def cond_dist(node_var, sampled_ctx):
    """Retrieve state distribution given sampled parents.""" # docstring
    if node_var == "Weather": # root node
        return p_w_prior # prior probabilities
    if node_var == "DayType": # root node
        return p_d_prior # prior probabilities
    par_key = tuple(sampled_ctx[p] for p in parent_map[node_var]) # extract parent states
    p_primary = first_prob_map[node_var][par_key] # primary probability
    state_0, state_1 = var_domains[node_var] # state pair
    return {state_0: p_primary, state_1: 1 - p_primary} # conditional distribution

In [6]:
def ancestral_sample(num_samples, rand_seed):
    gen = np.random.default_rng(rand_seed) # setup generator
    records = [] # accumulator
    for _ in range(num_samples): # iterate count
        inst = {} # current instance
        for node in node_order: # topological traverse
            dist_map = cond_dist(node, inst) # local probabilities
            possible_states, state_probs = list(dist_map.keys()), list(dist_map.values()) # separate lists
            inst[node] = possible_states[gen.choice(len(possible_states), p=state_probs)] # sample node
        records.append(inst) # append record
    return pd.DataFrame(records, columns=node_order) # create dataframe

In [7]:
def learn(sample_df, fitting_strategy):
    learned_bn = DiscreteBayesianNetwork(graph_edges) # target structure
    if fitting_strategy == "MLE": # maximum likelihood
        estimator = MaximumLikelihoodEstimator(learned_bn, sample_df, state_names=var_domains) # mle estimator
        cpd_list = estimator.get_parameters() # estimate params
    else: # bayesian parameter learning
        estimator = BayesianEstimator(learned_bn, sample_df, state_names=var_domains) # bayesian estimator
        cpd_list = estimator.get_parameters(prior_type="BDeu", equivalent_sample_size=10) # bdeu params
    learned_bn.add_cpds(*cpd_list) # attach cpds
    learned_bn.check_model() # validate learned network
    return learned_bn # return model

In [8]:
def ac_table(bn_model):
    """Generate tidy table for AC CPD.""" # docstring
    ac_factor = bn_model.get_cpds("AC") # get ac factor
    entries = [] # row buffer
    for w_val in var_domains["Weather"]: # loop weather
        for o_val in var_domains["Occupancy"]: # loop occupancy
            entries.append(((w_val, o_val), float(ac_factor.get_value(AC="On", Weather=w_val, Occupancy=o_val)))) # extract table entry
    return pd.Series(dict(entries)) # return series

In [9]:
!pip install pgmpy
import os # system os
import warnings # warning module
warnings.filterwarnings("ignore") # suppress warnings
import itertools # combinatorial tools
import numpy as np # numerical backend
import pandas as pd # data handling
import networkx as nx # graph library
import matplotlib # plotting engine
matplotlib.use("Agg") # headless backend
import matplotlib.pyplot as plt # plot interface
from pgmpy.models import DiscreteBayesianNetwork # bayesian net
from pgmpy.factors.discrete import TabularCPD # discrete factor
from pgmpy.inference import VariableElimination # exact inference
from pgmpy.estimators import MaximumLikelihoodEstimator, BayesianEstimator # parameter learners

out_dir = "outputs_p1" # destination path
os.makedirs(out_dir, exist_ok=True) # create folder
rand_seed = 42 # reproducibility seed
pd.set_option("display.width", 200) # console width
pd.set_option("display.max_columns", 30) # max cols
var_domains = { # variable domain states
    "Weather":   ["Hot", "Mild", "Cold"], # weather states
    "DayType":   ["Weekday", "Weekend"], # day states
    "Occupancy": ["High", "Low"], # occupancy states
    "AC":        ["On", "Off"], # ac states
    "Load":      ["Heavy", "Light"], # load states
    "Energy":    ["High", "Low"], # energy states
} # domain map end
graph_edges = [("DayType", "Occupancy"), # network edges
               ("Weather", "AC"), ("Occupancy", "AC"), # ac parent links
               ("Occupancy", "Load"), # occupancy load link
               ("AC", "Energy"), ("Load", "Energy")] # energy parent links
node_order = ["Weather", "DayType", "Occupancy", "AC", "Load", "Energy"] # topological sequence

p_w_prior = {"Hot": 0.50, "Mild": 0.30, "Cold": 0.20} # weather prior
p_d_prior = {"Weekday": 0.70, "Weekend": 0.30} # day prior
p_occ_hi = {("Weekday",): 0.45, ("Weekend",): 0.80} # p occ high
p_ac_on = {("Hot", "High"): 0.90, ("Hot", "Low"): 0.60, # ac on warm
           ("Mild", "High"): 0.40, ("Mild", "Low"): 0.15, # ac on mild
           ("Cold", "High"): 0.10, ("Cold", "Low"): 0.05} # ac on cold
p_ld_hvy = {("High",): 0.75, ("Low",): 0.35} # load heavy
p_en_hi = {("On", "Heavy"): 0.92, ("On", "Light"): 0.68, # energy high on
           ("Off", "Heavy"): 0.55, ("Off", "Light"): 0.12} # energy high off

parent_map = { # structure adjacency
    "Weather": [], "DayType": [], # root nodes
    "Occupancy": ["DayType"], # single parent
    "AC": ["Weather", "Occupancy"], # dual parents
    "Load": ["Occupancy"], # single parent
    "Energy": ["AC", "Load"], # dual parents
} # end parent map
first_prob_map = { # conditional base table
    "Occupancy": p_occ_hi, "AC": p_ac_on, "Load": p_ld_hvy, "Energy": p_en_hi, # base state mapping
} # table map end

exp_net = build_expert_model() # construct network

print("=" * 70, "\nPART A  -  EXPERT NETWORK\n" + "=" * 70) # section header
assert exp_net.check_model(), "Model check failed" # model validation
for factor in exp_net.get_cpds(): # inspect cpds
    col_sums = factor.get_values().sum(axis=0) # marginal sums
    assert np.allclose(col_sums, 1.0), f"CPT of {factor.variable} does not sum to 1" # verify validity
    print(f"  CPT {factor.variable:<10s} OK  (all {len(col_sums)} parent configs sum to 1)") # log cpd status

draw_network(exp_net, f"{out_dir}/bayesian_network.png", "Smart-Home Energy Bayesian Network") # export visual

query_cases = { # inference definitions
    "Q1": ("Energy",  "High", {}), # prior energy
    "Q2": ("Energy",  "High", {"Weather": "Hot"}), # weather condition
    "Q3": ("Energy",  "High", {"DayType": "Weekend"}), # day condition
    "Q4": ("Energy",  "High", {"Weather": "Cold", "Occupancy": "High"}), # joint condition
    "Q5": ("Weather", "Hot",  {"Energy": "High"}), # inverted query
    "Q6": ("AC",      "On",   {"Energy": "High"}), # explanation query
    "Q7": ("AC",      "On",   {"Energy": "High", "Load": "Heavy"}), # explaining away
} # query end
query_labels = { # descriptive labels
    "Q1": "P(E=High)", # label q1
    "Q2": "P(E=High | W=Hot)", # label q2
    "Q3": "P(E=High | D=Weekend)", # label q3
    "Q4": "P(E=High | W=Cold, O=High)", # label q4
    "Q5": "P(W=Hot | E=High)", # label q5
    "Q6": "P(AC=On | E=High)", # label q6
    "Q7": "P(AC=On | E=High, L=Heavy)", # label q7
} # label end

exp_out = run_queries(exp_net) # execute queries
print("\nExpert model inference:") # print label
for q_k, q_val in exp_out.items(): # loop outputs
    print(f"  {q_k}: {query_labels[q_k]:<32s} = {q_val:.4f}") # display inference

print("\nExplaining away:") # explaining away header
print(f"  Q6 P(AC=On | E=High)               = {exp_out['Q6']:.4f}") # baseline effect
print(f"  Q7 P(AC=On | E=High, Load=Heavy)   = {exp_out['Q7']:.4f}") # conditioned effect
print(f"  Change (Q7 - Q6)                   = {exp_out['Q7'] - exp_out['Q6']:+.4f}") # delta output

print("\nQ1 by hand:") # analytical computation
p_occ_h_hand = sum(p_d_prior[d] * p_occ_hi[(d,)] for d in p_d_prior) # total p occ high
p_occ_marg = {"High": p_occ_h_hand, "Low": 1 - p_occ_h_hand} # occupancy marginals
print(f"  P(O=High) = 0.70*0.45 + 0.30*0.80 = {p_occ_h_hand:.4f};  P(O=Low) = {1 - p_occ_h_hand:.4f}") # print step
p_ac_by_occ = {} # ac distribution map
for o_st in ["High", "Low"]: # loop occupancy
    p_on_val = sum(p_w_prior[w] * p_ac_on[(w, o_st)] for w in p_w_prior) # weighted sum
    p_ac_by_occ[o_st] = {"On": p_on_val, "Off": 1 - p_on_val} # full ac dist
    print(f"  P(A=On | O={o_st}) = " + # log ac equation
          " + ".join(f"{p_w_prior[w]:.2f}*{p_ac_on[(w, o_st)]:.2f}" for w in p_w_prior) + f" = {p_on_val:.4f}") # log evaluation
p_e_by_occ = {} # energy map
manual_tot = 0.0 # accumulator
for o_st in ["High", "Low"]: # iterate occupancy
    p_ld_val = p_ld_hvy[(o_st,)] # load probability
    sub_val = 0.0 # branch accumulator
    print(f"  --- O = {o_st} : sum over (A, L) of P(A|O) P(L|O) P(E=High|A,L)") # branch log
    for a_st in ["On", "Off"]: # iterate ac
        for l_st, p_ld_inst in [("Heavy", p_ld_val), ("Light", 1 - p_ld_val)]: # iterate load
            p_e_inst = p_en_hi[(a_st, l_st)] # base probability
            prod_term = p_ac_by_occ[o_st][a_st] * p_ld_inst * p_e_inst # joint product
            sub_val += prod_term # accumulate term
            print(f"      A={a_st:<3s} L={l_st:<5s}: {p_ac_by_occ[o_st][a_st]:.4f} * {p_ld_inst:.2f} * {p_e_inst:.2f} = {prod_term:.6f}") # trace term
    p_e_by_occ[o_st] = sub_val # store subtotal
    print(f"      P(E=High | O={o_st}) = {sub_val:.6f}") # log conditional
    manual_tot += p_occ_marg[o_st] * sub_val # marginalize out occ
print(f"  P(E=High) = {p_occ_marg['High']:.3f}*{p_e_by_occ['High']:.6f} + " # report hand math
      f"{p_occ_marg['Low']:.3f}*{p_e_by_occ['Low']:.6f} = {manual_tot:.6f}") # print total
print(f"  Code (Q1)                          = {exp_out['Q1']:.6f}") # report code result
assert abs(manual_tot - exp_out["Q1"]) < 1e-9 # verify equality
print("  MATCH: hand calculation == code") # verification logged

print("\n" + "=" * 70, "\nPART B  -  d-SEPARATION\n" + "=" * 70) # section b header
print("Independence statements via pgmpy d-separation:") # intro label
dsep_queries = [ # independency statements
    ("Weather", "DayType", []), # independent pair
    ("Weather", "DayType", ["Energy"]), # active v structure
    ("Weather", "Load", ["Occupancy"]), # blocked path
] # test list end
for node_x, node_y, cond_z in dsep_queries: # loop tests
    is_sep = not exp_net.is_dconnected(node_x, node_y, observed=cond_z) # evaluate separation
    cond_str = f" | {', '.join(cond_z)}" if cond_z else "" # format condition
    print(f"  {node_x} _||_ {node_y}{cond_str}  ->  {'HOLDS' if is_sep else 'DOES NOT HOLD'}") # report result

ve_engine = VariableElimination(exp_net) # inference engine
def calc_prob(var, target_state, ev_dict): # inference helper
    return float(ve_engine.query([var], evidence=ev_dict, show_progress=False).get_value(**{var: target_state})) # query single val

print("\nNumerical verification:") # numerical checks
print("  (1) Weather _||_ DayType") # case 1
print(f"      P(W=Hot)                         = {calc_prob('Weather', 'Hot', {}):.4f}") # baseline
print(f"      P(W=Hot | D=Weekend)             = {calc_prob('Weather', 'Hot', {'DayType': 'Weekend'}):.4f}") # conditioned
print("  (2) Weather _||_ DayType | Energy") # case 2
print(f"      P(W=Hot | E=High)                = {calc_prob('Weather', 'Hot', {'Energy': 'High'}):.4f}") # conditioning collider
print(f"      P(W=Hot | E=High, D=Weekend)     = {calc_prob('Weather', 'Hot', {'Energy': 'High', 'DayType': 'Weekend'}):.4f}") # dependent check
print("  (3) Weather _||_ Load | Occupancy") # case 3
print(f"      P(W=Hot | O=High)                = {calc_prob('Weather', 'Hot', {'Occupancy': 'High'}):.4f}") # condition intermediate
print(f"      P(W=Hot | O=High, L=Heavy)       = {calc_prob('Weather', 'Hot', {'Occupancy': 'High', 'Load': 'Heavy'}):.4f}") # blocked check

print("\n" + "=" * 70, f"\nPART C  -  ANCESTRAL SAMPLING (seed = {rand_seed})\n" + "=" * 70) # section c header

sample_counts = [500, 5000, 20000] # dataset sizes
data_store = {} # sample registry
for cnt in sample_counts: # loop sizes
    df_samp = ancestral_sample(cnt, rand_seed) # generate dataset
    data_store[cnt] = df_samp # save df
    print(f"\n--- Dataset n = {cnt} ---") # dataset header
    print("Empirical Weather frequency (true: Hot .50, Mild .30, Cold .20):") # empirical label
    print(df_samp["Weather"].value_counts(normalize=True).reindex(var_domains["Weather"]).round(4).to_string()) # value counts
    print("Counts per (Weather, Occupancy):") # crosstab label
    print(pd.crosstab(df_samp["Weather"], df_samp["Occupancy"]).reindex(var_domains["Weather"]).to_string()) # cross tabulation
    df_samp.to_csv(f"{out_dir}/samples_{cnt}.csv", index=False) # export csv

print("\n" + "=" * 70, "\nPART D  -  LEARNING\n" + "=" * 70) # section d header

fit_models, fit_res = {}, {} # result mappings
for cnt, cur_df in data_store.items(): # iterate datasets
    fit_models[cnt] = {"MLE": learn(cur_df, "MLE"), "BDeu": learn(cur_df, "BDeu")} # fit networks
    fit_res[cnt] = {method: run_queries(m_obj) for method, m_obj in fit_models[cnt].items()} # evaluate queries

    cpt_df = pd.DataFrame({ # build comparison
        "n(W,O)": cur_df.groupby(["Weather", "Occupancy"]).size().reindex( # empirical occurrences
            pd.MultiIndex.from_product([var_domains["Weather"], var_domains["Occupancy"]])).fillna(0).astype(int), # full combinations
        "Expert":  ac_table(exp_net), # ground truth
        "MLE":     ac_table(fit_models[cnt]["MLE"]), # mle table
        "BDeu":    ac_table(fit_models[cnt]["BDeu"]), # bdeu table
    }) # dataframe construct end
    cpt_df.index.names = ["Weather", "Occupancy"] # setup indices
    print(f"\nP(AC=On | Weather, Occupancy)  -  n = {cnt}") # print header
    print(cpt_df.round(4).to_string()) # display cpt

print("\n" + "=" * 70, "\nPART E  -  COMPARISON\n" + "=" * 70) # section e header
mad_scores = {"MLE": [], "BDeu": []} # tracking map
comp_dfs = {} # comparative frames
for cnt in sample_counts: # loop sample counts
    table_rows = [] # row storage
    for q_id in query_cases: # evaluate queries
        e_val = exp_out[q_id] # expert value
        ml_val = fit_res[cnt]["MLE"][q_id] # mle value
        bd_val = fit_res[cnt]["BDeu"][q_id] # bdeu value
        table_rows.append({"Query": q_id, "Expert": e_val, "MLE": ml_val, "Bayesian": bd_val, # summary dictionary
                           "|Diff| MLE": abs(e_val - ml_val), "|Diff| Bayesian": abs(e_val - bd_val)}) # absolute residuals
    res_df = pd.DataFrame(table_rows).set_index("Query") # convert dataframe
    comp_dfs[cnt] = res_df # store metrics
    mad_scores["MLE"].append(res_df["|Diff| MLE"].mean()) # record mle mad
    mad_scores["BDeu"].append(res_df["|Diff| Bayesian"].mean()) # record bdeu mad
    print(f"\n--- Dataset size {cnt} ---") # report dataset
    print(res_df.round(4).to_string()) # display metrics
    print(f"Mean |Diff|: MLE = {res_df['|Diff| MLE'].mean():.5f}   Bayesian = {res_df['|Diff| Bayesian'].mean():.5f}") # print mads
    res_df.round(5).to_csv(f"{out_dir}/comparison_{cnt}.csv") # save table

plt.figure(figsize=(7, 4.5)) # initialize figure
plt.plot(sample_counts, mad_scores["MLE"], "o-", label="MLE") # mle curve
plt.plot(sample_counts, mad_scores["BDeu"], "s-", label="Bayesian (BDeu, ESS=10)") # bayesian curve
plt.xscale("log") # log axis
plt.xticks(sample_counts, [str(s) for s in sample_counts]) # tick positions
plt.xlabel("Dataset size (log scale)") # axis label
plt.ylabel("Mean |difference| over Q1-Q7") # axis label
plt.title("Learning error vs. dataset size") # chart title
plt.grid(alpha=0.3) # render grid
plt.legend() # show legend
plt.savefig(f"{out_dir}/mean_abs_diff.png", dpi=150, bbox_inches="tight") # export image
plt.close() # cleanup figure

df_500 = comp_dfs[500] # extract n500 metrics
for score_col in ["|Diff| MLE", "|Diff| Bayesian"]: # check methods
    top_q = df_500[score_col].idxmax() # worst query
    print(f"\nLargest error at n=500 ({score_col}): {top_q}  {query_labels[top_q]}  error = {df_500.loc[top_q, score_col]:.4f}") # report worst
raw_500 = data_store[500] # n500 samples
print("Evidence frequency in the 500-sample dataset:") # support title
print(f"  (W=Cold, O=High)  rows: {((raw_500.Weather == 'Cold') & (raw_500.Occupancy == 'High')).sum()}") # cold high count
print(f"  (E=High)          rows: {(raw_500.Energy == 'High').sum()}") # high energy count
print(f"  (E=High, L=Heavy) rows: {((raw_500.Energy == 'High') & (raw_500.Load == 'Heavy')).sum()}") # joint evidence count
print(f"  (Cold, High, AC=On) rows: " # joint 3 way
      f"{((raw_500.Weather == 'Cold') & (raw_500.Occupancy == 'High') & (raw_500.AC == 'On')).sum()}") # count instances
print("\nFigures and CSVs saved in:", out_dir) # log output

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 12.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pgmpy]
PART A  -  EXPERT NETWORK
  CPT Weather    OK  (all 1 parent configs sum to 1)
  CPT DayType    OK  (all 1 parent configs sum to 1)
  CPT Occupancy  OK  (all 2 parent configs sum to 1)
  CPT AC         OK  (all 6 parent configs sum to 1)
  CPT Load       OK  (all 2 parent configs sum to 1)
  CPT Energy     OK  (all 4 parent configs sum to 1)

Expert model inference:
  Q1: P(E=High)                        = 0.5806
  Q2: P(E=High | W=Hot)                = 0.7063
  Q3: P(E=High | D=Weekend)            = 0.6402
  Q4: P(E=High | W=Cold, O=High)       = 0.4843
  Q5: P(W=Hot | E=High)                = 0.6082
  Q6: P(AC=On | E=High)                = 0.6929
  Q7: P(AC=On | E=High, L=Heavy)       = 0.6499

Explaining away:
  Q6 P(AC=On | E=High)               = 0.6929
  Q7 P(AC=On | E=High, Load=Heavy)   = 0.6499
  Change (Q7 - Q6)                   = -0.0430

Q1 by h